<a href="https://colab.research.google.com/github/Kathyayini26/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kathyayini26/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My baseline rule – Content Opportunity Scoring

My rule identifies webpages that may benefit from a content review based on their Google Search Console performance.

It considers search impression volume and the relationship between clicks, impressions and average search position.

Pages with sufficient impressions but relatively low click-through rates, despite having reasonable search positions, will receive higher priority for a CTR improvement review.

**Reason codes:**
- `HIGH_IMPRESSIONS_LOW_CTR`: High search visibility but relatively low click-through rate.
- `LOW_IMPRESSIONS`: Limited search visibility; further investigation may be needed.
- `INSUFFICIENT_DATA`: Not enough reliable search data to make a recommendation.

**Action label:** `REVIEW_CONTENT_CTR`

The score is a prioritisation score for human review, not a prediction of guaranteed traffic improvement.

In [ ]:

# ============================================================
# WEEK 4 - SECTION 1: MY RULE AND REASON CODES
# Lane: Content Opportunity Scoring
# ============================================================

# Define the baseline rule
rule_name = "Content Opportunity Scoring"

reason_codes = {
    "HIGH_IMPRESSIONS_LOW_CTR":
        "High impressions but relatively low CTR",

    "LOW_IMPRESSIONS":
        "Limited search visibility",

    "INSUFFICIENT_DATA":
        "Insufficient reliable search data"
}

action_label = "REVIEW_CONTENT_CTR"

print("BASELINE RULE")
print("-" * 50)
print("Rule:", rule_name)

print("\nREASON CODES")
for code, description in reason_codes.items():
    print(code, ":", description)

print("\nACTION LABEL:", action_label)


BASELINE RULE
--------------------------------------------------
Rule: Content Opportunity Scoring

REASON CODES
HIGH_IMPRESSIONS_LOW_CTR : High impressions but relatively low CTR
LOW_IMPRESSIONS : Limited search visibility
INSUFFICIENT_DATA : Insufficient reliable search data

ACTION LABEL: REVIEW_CONTENT_CTR


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### 2. Build the Ranked Queue

**Data source:** Hugging Face – `FlyRank/internship-warehouse`

**Table:** `fact_content_daily_performance`

**Partition:** March 2026 (`month=2026-03`)

I use the actual Hugging Face warehouse data and DuckDB to build a baseline action-scoring queue.

The analysis uses the following Google Search Console fields:

- `gsc_impressions` – number of search impressions.
- `gsc_clicks` – number of clicks received.
- `gsc_avg_position` – average search position.
- `gsc_data_available` – confirms whether Google Search Console data is available.

Only records where `gsc_data_available IS TRUE` are considered.

**Scoring approach:**

1. Calculate click-through rate (CTR) using clicks divided by impressions.
2. Identify high-impression pages using the 75th percentile of impressions.
3. Identify relatively low-CTR pages using the 25th percentile of CTR.
4. Give higher priority to pages with high impressions, low CTR and an average search position within the first 20 positions.
5. Assign a reason code and action label to each record.
6. Rank the records by baseline score and export the ranked queue.

**Output:** `work/outputs/baseline_action_score.csv`

This is a rule-based baseline for prioritising content reviews. The score indicates review priority; it does not guarantee future traffic improvement.

In [ ]:

# ============================================================
# WEEK 4 - SECTION 2: BUILD THE RANKED QUEUE
# Dataset: FlyRank/internship-warehouse
# Table: fact_content_daily_performance
# Partition: March 2026
# ============================================================

# STEP 1: INSTALL AND IMPORT LIBRARIES

%pip -q install -U duckdb huggingface_hub

import duckdb
import pandas as pd
import getpass
import os

print("Libraries imported successfully.")


# STEP 2: AUTHENTICATE WITH HUGGING FACE

HF_TOKEN = getpass.getpass("Enter your Hugging Face READ token: ")

if not HF_TOKEN:
    raise ValueError("Hugging Face token is required.")

con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
""")

print("Hugging Face authentication configured.")


# STEP 3: SELECT THE ACTUAL HUGGING FACE DATASET

dataset_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/**/*.parquet"
)

print("\nDataset:", "FlyRank/internship-warehouse")
print("Table:", "fact_content_daily_performance")
print("Partition:", "2026-03")


# STEP 4: CHECK THE DATA

data_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT report_date) AS number_of_dates,
        COUNT(DISTINCT client_hash_id) AS number_of_clients,
        COUNT(DISTINCT content_hash_id) AS number_of_contents
    FROM read_parquet(
        '{dataset_path}',
        hive_partitioning = true
    )
    WHERE gsc_data_available IS TRUE
""").df()

print("\nDATA CHECK")
display(data_check)


# STEP 5: AGGREGATE DATA BY CLIENT AND CONTENT

# The original data has daily records.
# We aggregate March data to avoid ranking the same
# client-content combination separately for every day.

aggregation_query = f"""
CREATE OR REPLACE TEMP VIEW content_summary AS

SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS total_impressions,

    SUM(gsc_clicks) AS total_clicks,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_clicks) * 1.0
             / SUM(gsc_impressions)
        ELSE NULL
    END AS ctr,

    CASE
        WHEN SUM(
            CASE
                WHEN gsc_avg_position IS NOT NULL
                THEN gsc_impressions
                ELSE 0
            END
        ) > 0
        THEN
            SUM(
                CASE
                    WHEN gsc_avg_position IS NOT NULL
                    THEN gsc_avg_position * gsc_impressions
                    ELSE 0
                END
            ) * 1.0
            /
            SUM(
                CASE
                    WHEN gsc_avg_position IS NOT NULL
                    THEN gsc_impressions
                    ELSE 0
                END
            )
        ELSE NULL
    END AS weighted_avg_position,

    COUNT(DISTINCT report_date) AS observed_days

FROM read_parquet(
    '{dataset_path}',
    hive_partitioning = true
)

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
"""

con.execute(aggregation_query)

print("Monthly content aggregation completed.")


# STEP 6: CALCULATE DATA-BASED THRESHOLDS

thresholds = con.execute("""
    SELECT
        quantile_cont(total_impressions, 0.75)
            AS high_impressions_threshold,

        quantile_cont(ctr, 0.25)
            AS low_ctr_threshold

    FROM content_summary

    WHERE total_impressions > 0
      AND ctr IS NOT NULL
      AND weighted_avg_position IS NOT NULL
""").fetchone()

high_impressions = thresholds[0]
low_ctr = thresholds[1]

if high_impressions is None or low_ctr is None:
    raise ValueError(
        "Could not calculate thresholds. Check the available data."
    )

print("\nCALCULATED THRESHOLDS")
print("--------------------------------")
print("High impressions threshold:", high_impressions)
print("Low CTR threshold:", low_ctr)


# STEP 7: BUILD THE BASELINE SCORE AND REASON CODES

# Score 100: High impressions + low CTR + position <= 20
# Score 50: Lower impression volume
# Score 0: Other records without a priority signal

scoring_query = f"""
CREATE OR REPLACE TEMP VIEW scored_content AS

SELECT
    client_hash_id,
    content_hash_id,
    total_impressions,
    total_clicks,
    ctr,
    weighted_avg_position,
    observed_days,

    CASE
        WHEN total_impressions >= {high_impressions}
         AND ctr <= {low_ctr}
         AND weighted_avg_position <= 20
            THEN 100

        WHEN total_impressions < {high_impressions}
            THEN 50

        ELSE 0
    END AS baseline_score,

    CASE
        WHEN total_impressions >= {high_impressions}
         AND ctr <= {low_ctr}
         AND weighted_avg_position <= 20
            THEN 'HIGH_IMPRESSIONS_LOW_CTR'

        WHEN total_impressions < {high_impressions}
            THEN 'LOW_IMPRESSIONS'

        ELSE 'NO_PRIORITY_SIGNAL'
    END AS reason_code,

    CASE
        WHEN total_impressions >= {high_impressions}
         AND ctr <= {low_ctr}
         AND weighted_avg_position <= 20
            THEN 'REVIEW_CONTENT_CTR'

        WHEN total_impressions < {high_impressions}
            THEN 'INVESTIGATE_VISIBILITY'

        ELSE 'NO_ACTION'
    END AS action_label

FROM content_summary
"""

con.execute(scoring_query)

print("\nBaseline scoring completed.")


# STEP 8: RANK THE CONTENT OPPORTUNITIES

con.execute("""
CREATE OR REPLACE TEMP VIEW ranked_queue AS

SELECT
    ROW_NUMBER() OVER (
        ORDER BY
            baseline_score DESC,
            total_impressions DESC,
            ctr ASC
    ) AS rank,

    *

FROM scored_content
""")


# STEP 9: CREATE THE OUTPUT DIRECTORY

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"


# STEP 10: EXPORT THE RANKED QUEUE TO CSV

con.execute(f"""
    COPY (
        SELECT *
        FROM ranked_queue
        ORDER BY rank
    )
    TO '{output_path}'
    (HEADER, DELIMITER ',')
""")

print("\nCSV FILE CREATED SUCCESSFULLY")
print("Output location:", output_path)


# STEP 11: DISPLAY THE TOP 20 RECORDS

top_20 = con.execute("""
    SELECT *
    FROM ranked_queue
    ORDER BY rank
    LIMIT 20
""").df()

print("\nTOP 20 CONTENT OPPORTUNITIES")
display(top_20)


# STEP 12: CHECK OUTPUT SUMMARY

summary = con.execute("""
    SELECT
        reason_code,
        COUNT(*) AS number_of_records,
        AVG(baseline_score) AS average_score
    FROM ranked_queue
    GROUP BY reason_code
    ORDER BY number_of_records DESC
""").df()

print("\nREASON CODE SUMMARY")
display(summary)


# STEP 13: VERIFY THE EXPORTED CSV

csv_check = con.execute(f"""
    SELECT COUNT(*)
    FROM read_csv_auto('{output_path}')
""").fetchone()[0]

print("\nFINAL VERIFICATION")
print("--------------------------------")
print("Total ranked records:", csv_check)
print("Top 20 displayed:", len(top_20))
print("CSV exists:", os.path.exists(output_path))
print("CSV size (MB):", round(os.path.getsize(output_path) / (1024 * 1024), 2))

print("\nSECTION 2 COMPLETED SUCCESSFULLY.")


Libraries imported successfully.
Enter your Hugging Face READ token: ··········
Hugging Face authentication configured.

Dataset: FlyRank/internship-warehouse
Table: fact_content_daily_performance
Partition: 2026-03


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


DATA CHECK


,total_rows,number_of_dates,number_of_clients,number_of_contents
0,3611061,31,47,176738


Monthly content aggregation completed.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (6 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


REASON CODE SUMMARY


,reason_code,number_of_records,average_score
0,LOW_IMPRESSIONS,132549,50.0
1,NO_PRIORITY_SIGNAL,41494,0.0
2,HIGH_IMPRESSIONS_LOW_CTR,2695,100.0



FINAL VERIFICATION
--------------------------------
Total ranked records: 176738
Top 20 displayed: 20
CSV exists: True
CSV size (MB): 21.82

SECTION 2 COMPLETED SUCCESSFULLY.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### 3. Top-20 Review

I reviewed the top 20 records generated by my baseline action-scoring rule using the March 2026 Hugging Face warehouse data.

For each record, I examined its ranking, score, reason code and recommended action.

The confidence note explains how strongly the available data supports the recommendation. The limitation explains what additional information or data-quality issue could make the recommendation incorrect.

These recommendations are intended to support human review, not automatically make content changes.

The review considers search impressions, clicks, CTR and weighted average search position. Since the analysis uses one month's data, the recommendations should be validated before implementation.

In [ ]:

# ============================================================
# WEEK 4 - SECTION 3: TOP-20 REVIEW
# Source: Hugging Face FlyRank warehouse
# ============================================================

# Step 1: Retrieve the top 20 ranked records

top_20_review = con.execute("""
    SELECT
        rank,
        client_hash_id,
        content_hash_id,
        total_impressions,
        total_clicks,
        ctr,
        weighted_avg_position,
        baseline_score,
        reason_code,
        action_label
    FROM ranked_queue
    ORDER BY rank
    LIMIT 20
""").df()


# Step 2: Add confidence notes and possible limitations

def confidence_note(row):
    if row["reason_code"] == "HIGH_IMPRESSIONS_LOW_CTR":
        return (
            "Moderate confidence: high impressions and low CTR "
            "support a CTR review."
        )

    elif row["reason_code"] == "LOW_IMPRESSIONS":
        return (
            "Low confidence: limited search visibility suggests "
            "investigation, but the cause is unknown."
        )

    else:
        return (
            "Low confidence: the available signals do not show "
            "a clear priority opportunity."
        )


def what_could_make_it_wrong(row):
    if row["reason_code"] == "HIGH_IMPRESSIONS_LOW_CTR":
        return (
            "Search intent, title relevance, SERP features or "
            "seasonality may explain the low CTR."
        )

    elif row["reason_code"] == "LOW_IMPRESSIONS":
        return (
            "Low demand, indexing issues, a new page or limited "
            "measurement coverage may explain low impressions."
        )

    else:
        return (
            "The selected signals may not capture the actual "
            "content opportunity."
        )


top_20_review["confidence_note"] = top_20_review.apply(
    confidence_note, axis=1
)

top_20_review["what_would_make_it_wrong"] = top_20_review.apply(
    what_could_make_it_wrong, axis=1
)


# Step 3: Display the complete review

print("TOP-20 CONTENT OPPORTUNITY REVIEW")
print("=" * 70)

display(top_20_review)


# Step 4: Check that all 20 records have review notes

assert len(top_20_review) == 20, (
    "Expected 20 records. Check the ranked queue."
)

assert top_20_review["confidence_note"].notna().all()
assert top_20_review["what_would_make_it_wrong"].notna().all()

print("\nREVIEW CHECK")
print("Records reviewed:", len(top_20_review))
print("Confidence notes completed:",
      top_20_review["confidence_note"].notna().sum())
print("Limitations completed:",
      top_20_review["what_would_make_it_wrong"].notna().sum())


# Step 5: Save the review as a separate CSV

review_path = "work/outputs/top20_review.csv"

top_20_review.to_csv(review_path, index=False)

print("\nTop-20 review saved to:", review_path)



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

TOP-20 CONTENT OPPORTUNITY REVIEW


,rank,client_hash_id,content_hash_id,total_impressions,total_clicks,ctr,weighted_avg_position,baseline_score,reason_code,action_label,confidence_note,what_would_make_it_wrong
0,1,client_23a62021009f63c4,content_bf078007df823490,44707.0,0.0,0.0,1.400049,100,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CONTENT_CTR,Moderate confidence: high impressions and low ...,"Search intent, title relevance, SERP features ..."
1,2,client_62f4a7e64f5e0096,content_0c5606abaaab3178,38865.0,0.0,0.0,4.804554,100,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CONTENT_CTR,Moderate confidence: high impressions and low ...,"Search intent, title relevance, SERP features ..."
2,3,client_73cda7b4e4f265ea,content_23a42776a7009b65,28950.0,0.0,0.0,9.563005,100,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CONTENT_CTR,Moderate confidence: high impressions and low ...,"Search intent, title relevance, SERP features ..."
3,4,client_e547b89c05043229,content_713b157e9c77690a,24908.0,0.0,0.0,3.431106,100,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CONTENT_CTR,Moderate confidence: high impressions and low ...,"Search intent, title relevance, SERP features ..."
4,5,client_73cda7b4e4f265ea,content_c9f840183215651b,21519.0,0.0,0.0,5.347182,100,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CONTENT_CTR,Moderate confidence: high impressions and low ...,"Search intent, title relevance, SERP features ..."
5,6,client_62f4a7e64f5e0096,content_17d994b99d470434,19938.0,0.0,0.0,10.909118,100,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CONTENT_CTR,Moderate confidence: high impressions and low ...,"Search intent, title relevance, SERP features ..."
6,7,client_3197e6291363b4db,content_93d76695da196fdf,19292.0,0.0,0.0,8.243572,100,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CONTENT_CTR,Moderate confidence: high impressions and low ...,"Search intent, title relevance, SERP features ..."
7,8,client_fef1a8f436438636,content_cc9732f0da1d8d2d,17115.0,0.0,0.0,6.012854,100,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CONTENT_CTR,Moderate confidence: high impressions and low ...,"Search intent, title relevance, SERP features ..."
8,9,client_1a730cb2640a1abf,content_b9d46abd9ffa6c8a,15902.0,0.0,0.0,8.564017,100,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CONTENT_CTR,Moderate confidence: high impressions and low ...,"Search intent, title relevance, SERP features ..."
9,10,client_23a62021009f63c4,content_fe8baba849843607,14813.0,0.0,0.0,3.130493,100,HIGH_IMPRESSIONS_LOW_CTR,REVIEW_CONTENT_CTR,Moderate confidence: high impressions and low ...,"Search intent, title relevance, SERP features ..."



REVIEW CHECK
Records reviewed: 20
Confidence notes completed: 20
Limitations completed: 20

Top-20 review saved to: work/outputs/top20_review.csv


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### 4. Weak Picks and Leakage Check

The baseline recommendations are based on March 2026 Google Search Console data.

Some picks may be weak because:

- Low impressions do not necessarily mean that a page has poor content.
- Low CTR may be caused by search intent, competition or search result features.
- Average search position can vary over time.
- A single month of data may not represent long-term performance.

**Leakage check:**

The scoring rule uses only search impressions, clicks, CTR and average search position from the March 2026 partition.

It does not use future performance windows, product flags or future outcome labels as scoring inputs.

The recommendations are therefore intended as a baseline for human review and require further validation.

In [ ]:

# ============================================================
# WEEK 4 - SECTION 4: WEAK PICKS + LEAKAGE CHECK
# Dataset: FlyRank/internship-warehouse
# Table: fact_content_daily_performance
# Partition: March 2026
# ============================================================

import pandas as pd

# STEP 1: IDENTIFY WEAK PICKS

print("=" * 65)
print("STEP 1: WEAK PICKS")
print("=" * 65)

weak_picks = con.execute("""
    SELECT
        rank,
        client_hash_id,
        content_hash_id,
        total_impressions,
        total_clicks,
        ctr,
        weighted_avg_position,
        baseline_score,
        reason_code,
        action_label
    FROM ranked_queue
    WHERE baseline_score = 0
       OR reason_code = 'LOW_IMPRESSIONS'
    ORDER BY rank
    LIMIT 20
""").df()

if weak_picks.empty:
    print("No weak picks found.")
else:
    display(weak_picks)

print("\nNumber of weak picks displayed:", len(weak_picks))


# STEP 2: IDENTIFY THE ACTUAL SCORING INPUTS

print("\n" + "=" * 65)
print("STEP 2: SCORING INPUTS")
print("=" * 65)

scoring_inputs = [
    "total_impressions",
    "total_clicks",
    "ctr",
    "weighted_avg_position"
]

for column in scoring_inputs:
    print("-", column)


# STEP 3: IDENTIFY RULE OUTPUTS
# These are created by our scoring rule.
# They are NOT input features.

print("\n" + "=" * 65)
print("STEP 3: RULE OUTPUTS")
print("=" * 65)

output_columns = [
    "rank",
    "baseline_score",
    "reason_code",
    "action_label"
]

for column in output_columns:
    print("-", column)


# STEP 4: CHECK FOR FUTURE OR PRODUCT-FLAG INPUTS

print("\n" + "=" * 65)
print("STEP 4: LEAKAGE CHECK")
print("=" * 65)

restricted_terms = [
    "product_flag",
    "future",
    "next_",
    "outcome",
    "label"
]

possible_leakage = [
    column
    for column in scoring_inputs
    if any(term in column.lower() for term in restricted_terms)
]

print("Potential leakage inputs:", possible_leakage)

assert len(possible_leakage) == 0, (
    "Potential leakage found. Review the scoring inputs."
)

print("PASS: No future-window, product-flag or outcome "
      "columns are included in the defined scoring inputs.")


# STEP 5: VERIFY SCORING INPUTS EXIST IN THE SOURCE DATA

print("\n" + "=" * 65)
print("STEP 5: SOURCE DATA VERIFICATION")
print("=" * 65)

source_schema = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{dataset_path}',
        hive_partitioning = true
    )
""").df()

source_columns = source_schema["column_name"].tolist()

required_source_columns = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "gsc_data_available"
]

missing_columns = [
    column
    for column in required_source_columns
    if column not in source_columns
]

print("Required source columns:", required_source_columns)
print("Missing required columns:", missing_columns)

assert len(missing_columns) == 0, (
    "Some required source columns are missing."
)

print("PASS: Required Hugging Face source columns verified.")


# STEP 6: VERIFY THE SCORING DATA

print("\n" + "=" * 65)
print("STEP 6: SCORING DATA VERIFICATION")
print("=" * 65)

scoring_check = con.execute("""
    SELECT
        COUNT(*) AS total_records,
        COUNT(*) FILTER (
            WHERE total_impressions IS NULL
        ) AS missing_impressions,
        COUNT(*) FILTER (
            WHERE total_clicks IS NULL
        ) AS missing_clicks,
        COUNT(*) FILTER (
            WHERE ctr IS NULL
        ) AS missing_ctr,
        COUNT(*) FILTER (
            WHERE weighted_avg_position IS NULL
        ) AS missing_position
    FROM ranked_queue
""").df()

display(scoring_check)


# STEP 7: FINAL SUMMARY

print("\n" + "=" * 65)
print("FINAL SECTION 4 SUMMARY")
print("=" * 65)

print("Dataset: FlyRank/internship-warehouse")
print("Table: fact_content_daily_performance")
print("Partition: March 2026")

print("\nScoring inputs:")
print(", ".join(scoring_inputs))

print("\nRule outputs:")
print(", ".join(output_columns))

print("\nFuture-window or product-flag inputs found:",
      len(possible_leakage))

print("\nRequired source columns missing:",
      len(missing_columns))

print("\nWeak picks displayed:", len(weak_picks))

print("\nFINAL RESULT:")
if len(possible_leakage) == 0 and len(missing_columns) == 0:
    print("PASS - Defined scoring inputs contain no identified")
    print("future-window, product-flag or outcome columns.")
else:
    print("REVIEW REQUIRED.")

print("\nSECTION 4 COMPLETED.")




STEP 1: WEAK PICKS


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rank,client_hash_id,content_hash_id,total_impressions,total_clicks,ctr,weighted_avg_position,baseline_score,reason_code,action_label
0,2696,client_fef1a8f436438636,content_495f68807fd67729,1038.0,0.0,0.000000,33.579961,50,LOW_IMPRESSIONS,INVESTIGATE_VISIBILITY
1,2697,client_62f4a7e64f5e0096,content_334d998f32270d94,1038.0,0.0,0.000000,25.016378,50,LOW_IMPRESSIONS,INVESTIGATE_VISIBILITY
2,2698,client_2094c6eb080311d5,content_2544b0336dd4184a,1038.0,0.0,0.000000,58.266859,50,LOW_IMPRESSIONS,INVESTIGATE_VISIBILITY
3,2699,client_73cda7b4e4f265ea,content_f558392b20dc0844,1038.0,0.0,0.000000,6.228324,50,LOW_IMPRESSIONS,INVESTIGATE_VISIBILITY
4,2700,client_73cda7b4e4f265ea,content_46675baf36a78a0d,1038.0,0.0,0.000000,4.582852,50,LOW_IMPRESSIONS,INVESTIGATE_VISIBILITY
5,2701,client_e547b89c05043229,content_59ad08a0c2c817f3,1038.0,1.0,0.000963,8.294798,50,LOW_IMPRESSIONS,INVESTIGATE_VISIBILITY
6,2702,client_fef1a8f436438636,content_3cfe5318aebc4bcf,1038.0,1.0,0.000963,7.942197,50,LOW_IMPRESSIONS,INVESTIGATE_VISIBILITY
7,2703,client_e547b89c05043229,content_51aa5ce3d370539d,1038.0,1.0,0.000963,8.466281,50,LOW_IMPRESSIONS,INVESTIGATE_VISIBILITY
8,2704,client_fef1a8f436438636,content_7ac2fb7aa99eea64,1038.0,2.0,0.001927,5.268786,50,LOW_IMPRESSIONS,INVESTIGATE_VISIBILITY
9,2705,client_e547b89c05043229,content_253229ee0374c943,1038.0,2.0,0.001927,15.041426,50,LOW_IMPRESSIONS,INVESTIGATE_VISIBILITY



Number of weak picks displayed: 20

STEP 2: SCORING INPUTS
- total_impressions
- total_clicks
- ctr
- weighted_avg_position

STEP 3: RULE OUTPUTS
- rank
- baseline_score
- reason_code
- action_label

STEP 4: LEAKAGE CHECK
Potential leakage inputs: []
PASS: No future-window, product-flag or outcome columns are included in the defined scoring inputs.

STEP 5: SOURCE DATA VERIFICATION
Required source columns: ['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'gsc_data_available']
Missing required columns: []
PASS: Required Hugging Face source columns verified.

STEP 6: SCORING DATA VERIFICATION


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_records,missing_impressions,missing_clicks,missing_ctr,missing_position
0,176738,0,0,0,0



FINAL SECTION 4 SUMMARY
Dataset: FlyRank/internship-warehouse
Table: fact_content_daily_performance
Partition: March 2026

Scoring inputs:
total_impressions, total_clicks, ctr, weighted_avg_position

Rule outputs:
rank, baseline_score, reason_code, action_label

Future-window or product-flag inputs found: 0

Required source columns missing: 0

Weak picks displayed: 20

FINAL RESULT:
PASS - Defined scoring inputs contain no identified
future-window, product-flag or outcome columns.

SECTION 4 COMPLETED.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.